# GDP and Financial Uncertainty — source-only research archive

This notebook preserves the original exploratory analysis, with stored execution outputs removed for readability and to avoid presenting old outputs as newly verified results. It has **not** been rerun end-to-end during publication. The VAR/AR(1) comparison is exploratory, and later VARMAX and rolling-forecast cells may contain unresolved alignment errors. See [the research note](../gdp-uncertainty-var.md) for interpretation and limitations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas_datareader import data as pdr
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
import datetime

In [ ]:
start = datetime.datetime(2000, 1, 1)
end = datetime.datetime(2024, 12, 31)

In [ ]:
gdp = pdr.DataReader('GDPC1', 'fred', start, end)  # Real GDP, quarterly
vix = pdr.DataReader('VIXCLS', 'fred', start, end)  # VIX, daily

gdp = gdp.dropna()
vix = vix.dropna()

In [ ]:
# Quarterly Standard Deviation: VIX
vix_quarterly_std = vix.resample('Q').std()

# VIX to GDP Alignment: Matching Quarterly GDP Start Dates
vix_quarterly_std.index = vix_quarterly_std.index.to_period('Q').to_timestamp(how='start')

# Trim both datasets to the common index range.
common_index = gdp.index.intersection(vix_quarterly_std.index) # index.intersection --> Aligned timestamp-sorting (in case of missing quarters)
gdp = gdp.loc[common_index] # Finding and keeping just the common timestamps through x (GDP or VIX)
vix_quarterly_std = vix_quarterly_std.loc[common_index]

In [ ]:
data = pd.concat([gdp, vix_quarterly_std], axis=1)
data.columns = ['GDP', 'Uncertainty']
data = data.dropna()

print(data.head())
print(data.describe())

In [ ]:
data.plot(subplots=True, figsize=(10,6), title=['Real GDP', 'Uncertainty (VIX Rolling Std - Proxy)'])
plt.tight_layout()
plt.show()

VAR models assume stationarity (mean, variance, and covariance remain constant; the values deviate around a stable mean). But, obviously, predominantly, macro variables tend to be non-stationary. Therefore, we will need to difference them.
The key problem is that trends may obfuscate our analysis of the real dynamics of the system in the model. Therefore, we need to ensure stationarity through differencing.
We can already tell using visual inspection that GDP is non-stationary and will require differencing, and the uncertainty proxy will not.

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
def adf_test(series, title=''):
    print(f'ADF Test: {title}')
    result = adfuller(series.dropna(), autolag='AIC') # Akaike information criterion to avoid under (too few lags; high error) 
    # or overfitting (too many lags, or noise); to select the best model.
    labels = ['ADF Statistic', 'p-value', 'Lags Used', 'Observations']
    print('ADF Statistic:', result[0])
    print('p-value:', result[1])
    print('Lags Used:', result[2])
    
    if result[1] <= 0.05:
        print("Stationary (reject null hypothesis)")
    else:
        print("Non-stationary (accept null hypothesis)")
        print("...")

adf_test(data['GDP'], title='GDP')
adf_test(data['Uncertainty'], title='Uncertainty')

In [ ]:
# GDP, naturally (visually), is non-stationary.
data_diff = data.diff().dropna()
adf_test(data_diff['GDP'], title='GDP (First Difference)')

In [ ]:
plt.plot(data_diff)
plt.title('Differenced GDP')
plt.xlabel('Date')
plt.ylabel('ΔGDP')
plt.grid(True)
plt.show()

In [ ]:
print(data_diff.head())

In [ ]:
from statsmodels.tsa.api import VAR

model = VAR(data_diff)

# What lag order should we select?
lag_results = model.select_order(maxlags=8)
print(lag_results.summary())

In [ ]:
model_fitted = model.fit(5)

In [ ]:
var_model = VAR(data_diff[['GDP', 'Uncertainty']])
var_results = var_model.fit(5)

Regression Results

In [ ]:
print("GDP Equation Coefficients")
gdp_results = var_results.params['GDP']
gdp_se = var_results.stderr['GDP']
gdp_t = var_results.tvalues['GDP']
gdp_p = var_results.pvalues['GDP']

gdp_table = pd.DataFrame({
    'Coefficient': gdp_results,
    'Std. Error': gdp_se,
    't-Statistic': gdp_t,
    'p-Value': gdp_p
})
print(gdp_table)

\begin{aligned}
GDP_t &= \alpha_1 + \beta_{11} GDP_{t-1} + \beta_{12} GDP_{t-2} + \beta_{13} GDP_{t-3} + \beta_{14} GDP_{t-4} + \beta_{15} GDP_{t-5} + \gamma_{11} U_{t-1} + \gamma_{12} U_{t-2} + \gamma_{13} U_{t-3} + \gamma_{14} U_{t-4} + \gamma_{15} U_{t-5} + \varepsilon_{1t}
\\
GDP_t &= 123.83\, - 0.208\, GDP_{t-1} - 0.008\, GDP_{t-2} + 0.006\, GDP_{t-3} - 0.199\, GDP_{t-4} + 0.154\, GDP_{t-5} - 67.93\, U_{t-1} - 24.66\, U_{t-2} - 23.33\, U_{t-3} - 16.90\, U_{t-4} - 27.62\, U_{t-5} + \varepsilon_{1t}
\end{aligned}

In [ ]:
print("Uncertainty Equation")
uncertainty_results = var_results.params['Uncertainty']
uncertainty_se = var_results.stderr['Uncertainty']
uncertainty_t = var_results.tvalues['Uncertainty']
uncertainty_p = var_results.pvalues['Uncertainty']

uncertainty_table = pd.DataFrame({
    'Coefficient': uncertainty_results,
    'Std. Error': uncertainty_se,
    't-Statistic': uncertainty_t,
    'p-Value': uncertainty_p
})
print(uncertainty_table)

\begin{aligned}
U_t &= \alpha_2 + \beta_{21} GDP_{t-1} + \beta_{22} GDP_{t-2} + \beta_{23} GDP_{t-3} + \beta_{24} GDP_{t-4} + \beta_{25} GDP_{t-5} + \gamma_{21} U_{t-1} + \gamma_{22} U_{t-2} + \gamma_{23} U_{t-3} + \gamma_{24} U_{t-4} + \gamma_{25} U_{t-5} + \varepsilon_{2t}
\\
U_t &= -0.741\, + 0.0016\, GDP_{t-1} + 0.0012\, GDP_{t-2} - 0.0016\, GDP_{t-3} + 0.0042\, GDP_{t-4} + 0.0024\, GDP_{t-5} - 0.6565\, U_{t-1} - 0.4868\, U_{t-2} - 0.4158\, U_{t-3} - 0.4454\, U_{t-4} + 0.1106\, U_{t-5} + \varepsilon_{2t}
\end{aligned}

In [ ]:
print("Model Fit Statistics")
print(f"AIC:  {var_results.aic}")
print(f"BIC:  {var_results.bic}")
print(f"HQIC: {var_results.hqic}")
print(f"FPE:  {var_results.fpe}")
print(f"Log Likelihood: {var_results.llf}")

In [ ]:
print("Residual Correlation Matrix")
print(var_results.resid.corr())

In [ ]:
# Impulse response of GDP to a shock in Uncertainty
irf = var_results.irf(10)  # The horizon is 10 quarters
irf.plot(orth=False, signif=0.05)

Initial Analysis (Visual Inspection):
The IRF shows how a unit shock in one variable affects the complete system over time.

1) A unit shock in GDP to overall GDP: Standard autoregressive behaviour
2) A unit shock in Uncertainty to overall GDP: Most important observation - GDP contracts when uncertainty rises
3) A unit shock in GDP to overall Uncertainty: Weak effect (Slightly positive drift?)
4) A unit shock in Uncertainty to overall Uncertainty: Mean reversion - Significant drop, then recovery

In [ ]:
# GDP --> GDP
irf.plot(impulse='GDP', response='GDP', figsize=(6, 4))
plt.title('IRF: GDP Shock → GDP Response')
plt.tight_layout()
plt.show()

In [ ]:
# Uncertainty --> GDP
irf.plot(impulse='Uncertainty', response='GDP', figsize=(6, 4))
plt.title('IRF: Uncertainty Shock → GDP Response')
plt.tight_layout()
plt.show()

In [ ]:
# GDP --> Uncertainty
irf.plot(impulse='GDP', response='Uncertainty', figsize=(6, 4))
plt.title('IRF: GDP Shock → Uncertainty Response')
plt.tight_layout()
plt.show()

In [ ]:
# Uncertainty --> Uncertainty
irf.plot(impulse='Uncertainty', response='Uncertainty', figsize=(6, 4))
plt.title('IRF: Uncertainty Shock → Uncertainty Response')
plt.tight_layout()
plt.show()

In [ ]:
# Granger causality test
max_lag = 5
granger_results = var_results.test_causality('GDP', ['Uncertainty'], kind='f')
print(granger_results.summary())

granger_results_rev = var_results.test_causality('Uncertainty', ['GDP'], kind='f')
print(granger_results_rev.summary())

These findings justify policies that treat uncertainty as an exogenous variable when modelling real economic fluctuations. For instance, this highlights the imperative nature of information asymmetry and stabilising expectations (central banks could prioritise clear communication and forward guidance).

In [ ]:
fevd = var_results.fevd(10)  # Forecast error variance decomposition -- 10 quarters ahead
fevd.summary()

In [ ]:
# Step 1: Forecasting
forecast_steps = 8
forecast = var_results.forecast(data_diff.values[-var_results.k_ar:], steps=forecast_steps)

# Step 2: Create DataFrame for readability
forecast_index = pd.date_range(start=data_diff.index[-1] + pd.DateOffset(months=3),
                               periods=forecast_steps, freq='Q')
forecast_df = pd.DataFrame(forecast, index=forecast_index, columns=['GDP_forecast', 'Uncertainty_forecast'])

# Step 3: Display forecast
print(forecast_df)

# Step 4: Plot
plt.figure(figsize=(10, 6))
plt.plot(data_diff['GDP'], label='GDP (Historical)')
plt.plot(forecast_df['GDP_forecast'], label='GDP (Forecast)', linestyle='--')
plt.title('GDP Forecast (4 Quarters Ahead)')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(data_diff['Uncertainty'], label='Uncertainty (Historical)')
plt.plot(forecast_df['Uncertainty_forecast'], label='Uncertainty (Forecast)', linestyle='--')
plt.title('Uncertainty Forecast (4 Quarters Ahead)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
ar_model = AutoReg(data_diff["GDP"], lags=1).fit()

In [ ]:
ar_forecast = ar_model.predict(start=len(data_diff), end=len(data_diff)+7)

In [ ]:
ar_forecast.index = forecast_df.index

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(data_diff['GDP'], label='GDP (Historical)')
plt.plot(forecast_df['GDP_forecast'], label='VAR Forecast', linestyle='--', marker='x')
plt.plot(ar_forecast, label='AR(1) Forecast', linestyle=':', marker='o')
plt.title("GDP Forecast: VAR vs AR(1)")
plt.xlabel("Forecast Step")
plt.ylabel("Differenced GDP")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print("Length of data_diff:", len(data_diff))

In [ ]:
from statsmodels.tsa.api import VAR
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error

# Split train-test
p = 5  # VAR lag order
train = data_diff[:-8]
test = data_diff[-8:]

model = VAR(train)
var_result = model.fit(p)

var_forecast = var_result.forecast(train.values[-p:], steps=8)
var_gdp_forecast = var_forecast[:, 0]

ar_model = AutoReg(train['GDP'], lags=1).fit()
ar_forecast = ar_model.predict(start=len(train), end=len(train) + 7)

plt.figure(figsize=(10, 5))
plt.plot(test.index, test['GDP'].values, label='True GDP', marker='o')
plt.plot(test.index, var_gdp_forecast, label='VAR Forecast', marker='x')
plt.plot(test.index, ar_forecast, label='AR(1) Forecast', marker='s')
plt.title("Forecast Comparison: VAR vs AR(1)")
plt.xlabel("Quarter")
plt.ylabel("GDP (Differenced)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("RMSE – VAR:", np.sqrt(mean_squared_error(test['GDP'], var_gdp_forecast)))
print("RMSE – AR(1):", np.sqrt(mean_squared_error(test['GDP'], ar_forecast)))

In [ ]:
data['GDP_log'] = np.log(data['GDP'])
data_diff['GDP_log'] = data['GDP_log'].diff().dropna()

In [ ]:
# If you've differenced your main variables
gdp_diff = gdp.diff().dropna()
vix = vix.dropna()

# Align time index and create DataFrame
data_diff = pd.DataFrame({
    'GDP_diff': gdp_diff.squeeze(),
    'VIX': vix.squeeze()
})

# Drop any NaNs that arise from alignment
data_diff_aligned = data_diff.dropna()

# (Optional) If you’re adding more exogenous variables, include here
# For now, just confirm shape
print("data_diff_aligned shape:", data_diff_aligned.shape)

In [ ]:
print("exog_data_aligned shape:", exog_data_aligned.shape)

In [ ]:
gdp_diff = gdp.diff().dropna()
ar_model = sm.tsa.ARIMA(gdp_diff, order=(1, 0, 0)).fit()

VARMAX to improve forecast accuracy

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pandas as pd

# STEP 1: Differenced GDP series (assuming `gdp` already exists)
gdp_diff = gdp.diff().dropna()

# STEP 2: Fit AR(1) model
ar_model = sm.tsa.ARIMA(gdp_diff, order=(1, 0, 0)).fit()

# STEP 3: Forecast next 10 quarters
forecast_steps = 10
forecast_index = pd.date_range(start=gdp_diff.index[-1] + pd.offsets.QuarterEnd(), periods=forecast_steps, freq='Q')
ar_forecast = ar_model.forecast(steps=forecast_steps)
ar_forecast_df = pd.DataFrame(ar_forecast, index=forecast_index, columns=["AR(1) Forecast"])

# STEP 4: Plot forecast
plt.figure(figsize=(10, 5))
plt.plot(gdp_diff[-20:], label="Actual (last 20)", color="black")
plt.plot(ar_forecast_df, label="AR(1) Forecast", linestyle="--", color="blue")
plt.title("AR(1) Benchmark Forecast for Differenced GDP")
plt.xlabel("Date")
plt.ylabel("ΔGDP")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# STEP 5: Model summary (optional)
print(ar_model.summary())

In [ ]:
# Replace index with valid quarterly range to avoid freq assignment errors
data_diff_aligned.index = pd.date_range(start='2001-01-01', periods=len(data_diff_aligned), freq='Q')
exog_data_aligned.index = pd.date_range(start='2001-01-01', periods=len(exog_data_aligned), freq='Q')

In [ ]:
from statsmodels.tsa.statespace.varmax import VARMAX

model = VARMAX(data_diff_aligned, exog=exog_data_aligned, order=(5, 0))
varmax_results = model.fit(disp=False)

print(varmax_results.summary())

In [ ]:
from statsmodels.tsa.statespace.varmax import VARMAX

# Step 1: Fit the VARMAX model
model = VARMAX(data_diff_aligned, exog=exog_data_aligned, order=(5, 0))
varmax_results = model.fit(disp=False)

# Step 2: Summary and diagnostics
print(varmax_results.summary())
varmax_results.plot_diagnostics(figsize=(12, 8))

In [ ]:
# Align exog_data_aligned to match the shorter differenced endogenous data
exog_data_aligned = exog_data_aligned.iloc[-data_diff_aligned.shape[0]:].copy()

# Final check
print("Endogenous (Y) shape:", data_diff_aligned.shape)
print("Exogenous (X) shape:", exog_data_aligned.shape)

In [ ]:
print(forecast_df.columns)

In [ ]:
# Forecasting 10 steps ahead
forecast = varmax_results.get_forecast(steps=10, exog=exog_data_aligned[-10:])
forecast_df = forecast.predicted_mean

# Plot GDP forecast component
plt.figure(figsize=(10, 5))
plt.plot(data_diff_aligned['GDP_diff'].iloc[-20:], label='Recent Actual (GDP)')
plt.plot(forecast_df['GDP_diff'], label='Forecasted GDP (VARMAX)', linestyle='--')
plt.title('VARMAX Forecast: Differenced GDP')
plt.xlabel('Date')
plt.ylabel('ΔGDP')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# STEP 1: Actual and predicted values
y_true = gdp_diff_actual.values
y_pred = gdp_diff_predicted.values

# STEP 2: Error metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# Safe MAPE: exclude 0 values
nonzero_mask = y_true != 0
mape = np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100

# STEP 3: Report
print("VARMAX Forecast Error Metrics:")
print(f"  MAE  = {mae:.4f}")
print(f"  RMSE = {rmse:.4f}")
print(f"  MAPE = {mape:.2f}%")

In [ ]:
window_size = 50                # number of observations to train on
forecast_horizon = 1           # how many steps ahead you predict each time
n_total = len(data_diff_aligned)

rolling_forecasts = []
rolling_actuals = []

In [ ]:
n_obs = len(rolling_forecasts)
rolling_index = data_diff_aligned.index[-n_obs:]  # take last n_obs dates

forecast_series = pd.Series(rolling_forecasts, index=rolling_index)
actual_series = pd.Series(rolling_actuals, index=rolling_index)

In [ ]:
# Assuming rolling_forecasts and rolling_actuals are your lists
n_obs = min(len(rolling_forecasts), len(rolling_actuals))
rolling_index = data_diff_aligned.index[-n_obs:]

forecast_series = pd.Series(rolling_forecasts[-n_obs:], index=rolling_index)
actual_series = pd.Series(rolling_actuals[-n_obs:], index=rolling_index)

In [ ]:
print(forecast.predicted_mean.columns)

In [ ]:
from statsmodels.tsa.statespace.varmax import VARMAX

for start in range(n_total - window_size - forecast_horizon + 1):
    end = start + window_size

    # Define rolling training sets
    y_train = data_diff_aligned.iloc[start:end]
    x_train = exog_data_aligned.iloc[start:end]
    x_forecast = exog_data_aligned.iloc[end:end+forecast_horizon]

    # Fit VARMAX
    try:
        model = VARMAX(y_train, exog=x_train, order=(1, 0))
        results = model.fit(disp=False)
    
        forecast = results.get_forecast(steps=forecast_horizon, exog=x_forecast)
        forecast_value = forecast.predicted_mean.iloc[0]['GDP_diff']
        actual_value = data_diff_aligned.iloc[end]['GDP_diff']
    
        # Append both only when both succeed
        rolling_forecasts.append(forecast_value)
        rolling_actuals.append(actual_value)
    
    except Exception as e:
        print(f"Skipped window {start}–{end} due to error: {e}")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(actual_series, forecast_series)
rmse = np.sqrt(mean_squared_error(actual_series, forecast_series))

print("ROLLING FORECAST ERROR:")
print(f"  MAE  = {mae:.4f}")
print(f"  RMSE = {rmse:.4f}")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(actual_series, label='Actual')
plt.plot(forecast_series, label='Rolling Forecast (VARMAX)', linestyle='--')
plt.title('Rolling 1-Step Forecasts: Differenced GDP')
plt.xlabel('Date')
plt.ylabel('ΔGDP')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
run_rolling_forecast(
    data=data_diff_aligned,
    exog=exog_data_aligned,
    target='GDP_diff',
    model_type='VARMAX',
    lags=1,
    window_size=50,
    forecast_horizon=1,
    use_exog=True,
    plot=True
)

In [ ]:
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.api import VAR
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def run_rolling_forecast(
    data,
    exog=None,
    target='GDP_diff',
    model_type='VARMAX',   # Options: 'VARMAX', 'VAR', 'AR'
    lags=1,
    window_size=50,
    forecast_horizon=1,
    use_exog=True,
    plot=True,
    verbose=True
):
    """
    Runs a rolling forecast on the selected model type.
    """

    rolling_forecasts = []
    rolling_actuals = []
    index_list = []

    n_total = len(data)

    for start in range(n_total - window_size - forecast_horizon + 1):
        end = start + window_size
        forecast_step = end + forecast_horizon - 1

        try:
            y_train = data.iloc[start:end]
            y_forecast_true = data.iloc[forecast_step][target]
            index_list.append(data.index[forecast_step])

            if model_type == 'VARMAX':
                if use_exog and exog is not None:
                    x_train = exog.iloc[start:end]
                    x_future = exog.iloc[forecast_step:forecast_step + 1]
                else:
                    x_train = None
                    x_future = None

                model = VARMAX(y_train, exog=x_train, order=(lags, 0))
                results = model.fit(disp=False)

                forecast = results.get_forecast(steps=1, exog=x_future)
                pred_value = forecast.predicted_mean.iloc[0][target]

            elif model_type == 'VAR':
                model = VAR(y_train)
                results = model.fit(lags)
                forecast = results.forecast(y_train.values[-lags:], steps=1)
                pred_value = forecast[0][y_train.columns.get_loc(target)]

            elif model_type == 'AR':
                model = AutoReg(y_train[target], lags=lags)
                results = model.fit()
                pred_value = results.predict(start=forecast_step, end=forecast_step)[0]

            else:
                raise ValueError("Model type must be one of: 'VARMAX', 'VAR', or 'AR'.")

            rolling_forecasts.append(pred_value)
            rolling_actuals.append(y_forecast_true)

        except Exception as e:
            if verbose:
                print(f"Skipped window {start}–{end} due to error: {e}")

    # Convert to Series
    forecast_series = pd.Series(rolling_forecasts, index=index_list)
    actual_series = pd.Series(rolling_actuals, index=index_list)

    # Evaluation
    mae = mean_absolute_error(actual_series, forecast_series)
    rmse = np.sqrt(mean_squared_error(actual_series, forecast_series))

    if verbose:
        print("\nROLLING FORECAST ERROR:")
        print(f"  MAE  = {mae:.4f}")
        print(f"  RMSE = {rmse:.4f}")

    # Plot
    if plot:
        plt.figure(figsize=(10, 5))
        plt.plot(actual_series, label='Actual')
        plt.plot(forecast_series, label=f'Rolling Forecast ({model_type})', linestyle='--')
        plt.title(f'Rolling 1-Step Forecast: {target}')
        plt.xlabel('Date')
        plt.ylabel(f'Δ{target}')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return actual_series, forecast_series, mae, rmse